<a href="https://colab.research.google.com/github/christo444/Dark-Guard/blob/main/new_roberta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers

import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- Step 1: upload your saved split file ---
from google.colab import files
print("Please upload darkguard_split_v1.tsv:")
uploaded = files.upload()
split_df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")

trainval_df = split_df[split_df["split"] == "trainval"].copy()
test_df = split_df[split_df["split"] == "test"].copy()
print(f"Original trainval pool: {len(trainval_df)} | Original held-out test: {len(test_df)}")

# --- Step 2: the 62 reviewed synthetic negatives, split 50 train / 12 OOD-eval ---
synthetic_negatives = [
    "Price: $24.99", "Price: ₹3,499", "$49.00", "₹899", "Rs. 1,250.00",
    "Total: $128.45", "Subtotal: $56.20", "$12.99 per item", "MRP: ₹2,999", "Regular price $75.00",
    "Quantity: 1", "Quantity:", "Qty: 2", "Item quantity", "Select quantity",
    "Update quantity", "1 item in cart", "3 items in your bag",
    "Friday, 25 September", "Order placed on March 3, 2024", "Delivery by June 12",
    "Estimated arrival: 2-3 business days", "Return window: 30 days", "Available from January 15", "Ships on Monday",
    "Delivering to Dindigul 624005", "Ship to: New York, NY 10001", "123 Main Street, Apt 4B",
    "Delivery address", "Change delivery address", "Shipping to United States", "Estimated delivery to your area",
    "Order #48213", "Tracking number: 1Z999AA10123456784", "Invoice #2024-0091",
    "Account number ending in 4417", "Reference ID: 88213",
    "30% off", "50% discount", "Save 20%", "10% cashback on this order", "Tax: 8.5%", "Interest rate: 4.9% APR",
    "4.5 out of 5 stars", "Rated 4.2 based on 310 reviews", "3 reviews", "128 customers rated this 5 stars",
    "Size: 8 UK", "Weight: 2.5 kg", "Dimensions: 12 x 8 x 4 inches", "Available in sizes 6, 8, 10, 12",
    "500ml bottle", "Pack of 3",
    "Call us at 1-800-555-0199", "Customer service: (555) 123-4567",
    "Page 2 of 14", "Showing 1-24 of 350 results", "12 items per page",
    "Card ending in 4417", "Expires 08/27", "3 installments of $16.33", "Pay in 4 interest-free payments",
]

random.seed(42)
shuffled = synthetic_negatives.copy()
random.shuffle(shuffled)
ood_eval_texts = shuffled[:12]
aug_train_texts = shuffled[12:]

aug_train_df = pd.DataFrame({
    "page_id": range(90000, 90000 + len(aug_train_texts)),
    "text": aug_train_texts,
    "label": 0,
    "Pattern Category": "Not Dark Pattern",
})
print(f"Added to training: {len(aug_train_df)} | Held out as new OOD test: {len(ood_eval_texts)}")

# --- Step 3: combine original trainval + synthetic negatives ---
combined_train_df = pd.concat([
    trainval_df[["page_id", "text", "label", "Pattern Category"]],
    aug_train_df
], ignore_index=True)
print(f"Combined augmented training pool: {len(combined_train_df)} rows")

# --- Step 4: tokenize everything ---
MAX_LENGTH = 64
tok = AutoTokenizer.from_pretrained("roberta-large")

def encode(texts):
    return tok(list(texts), padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_encodings = encode(combined_train_df["text"])
test_encodings = encode(test_df["text"])
ood_encodings = encode(ood_eval_texts)

backend_texts = [
    "Hurry! Only 1 left in stock!", "Only 4 left in stock.", "Only 2 items remaining!",
    "Selling fast! Only 3 left.", "23 people are viewing this right now.", "Offer ends in 10 minutes!",
    "LAST CHANCE - BUY NOW!", "Pillowcases & Shams", "Write a review", "International Shipping Policy",
    "Product Feature", "Sold by Amazon", "Quantity:", "Delivery date", "Friday, 25 September",
    "Delivering to Dindigul 624005", "3,499.00", "30% off", "50% discount", "Sale price ₹3,499",
    "Only 30% off", "Limited time 30% off", "30% off - offer ends tonight",
]
backend_labels = [1,1,1,1,1,1,1, 0,0,0,0,0,0,0,0,0,0,0,0,0,0, 1,1]
backend_encodings = encode(backend_texts)

class DarkPatternDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = DarkPatternDataset(train_encodings, list(combined_train_df["label"]))
test_dataset = DarkPatternDataset(test_encodings, list(test_df["label"]))
ood_dataset = DarkPatternDataset(ood_encodings, [0]*len(ood_eval_texts))
backend_dataset = DarkPatternDataset(backend_encodings, backend_labels)

# --- Step 5: train, same locked configuration as before ---
model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

training_args = TrainingArguments(
    output_dir="./roberta_large_augmented",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,
    fp16=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, compute_metrics=compute_metrics)
trainer.train()

# --- Step 6: three separate evaluations ---
print("\n=== 1. ORIGINAL 284-row held-out test (regression check) ===")
print(trainer.evaluate(test_dataset))

print("\n=== 2. NEW held-out OOD synthetic negatives (12 examples, all should predict 0) ===")
ood_preds = np.argmax(trainer.predict(ood_dataset).predictions, axis=1)
for text, pred in zip(ood_eval_texts, ood_preds):
    mark = "✓" if pred == 0 else "✗"
    print(f"{mark} predicted={pred} | {text}")
print(f"OOD accuracy (target=0 for all): {(ood_preds == 0).mean():.2%}")

print("\n=== 3. Backend team's original 23 real examples (independent sanity check) ===")
backend_preds = np.argmax(trainer.predict(backend_dataset).predictions, axis=1)
correct = 0
for text, pred, expected in zip(backend_texts, backend_preds, backend_labels):
    mark = "✓" if pred == expected else "✗"
    if pred == expected: correct += 1
    print(f"{mark} expected={expected} predicted={pred} | {text}")
print(f"Backend sanity accuracy: {correct}/{len(backend_texts)} = {correct/len(backend_texts):.2%}")
print("(Original, pre-fix result on this same set was 15/23 = 65.22%)")

Please upload darkguard_split_v1.tsv:


Saving darkguard_split_v1.tsv to darkguard_split_v1 (1).tsv
Original trainval pool: 2072 | Original held-out test: 284
Added to training: 50 | Held out as new OOD test: 12
Combined augmented training pool: 2122 rows


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.8953', 'grad_norm': '30.27', 'learning_rate': '1.816e-05', 'epoch': '0.3759'}
{'loss': '0.4831', 'grad_norm': '33.33', 'learning_rate': '1.628e-05', 'epoch': '0.7519'}
{'loss': '0.2317', 'grad_norm': '1.541', 'learning_rate': '1.44e-05', 'epoch': '1.128'}
{'loss': '0.3454', 'grad_norm': '0.2324', 'learning_rate': '1.252e-05', 'epoch': '1.504'}
{'loss': '0.2393', 'grad_norm': '134.5', 'learning_rate': '1.064e-05', 'epoch': '1.88'}
{'loss': '0.2108', 'grad_norm': '19.56', 'learning_rate': '8.759e-06', 'epoch': '2.256'}
{'loss': '0.1095', 'grad_norm': '0.03279', 'learning_rate': '6.88e-06', 'epoch': '2.632'}
{'loss': '0.05123', 'grad_norm': '0.2178', 'learning_rate': '5e-06', 'epoch': '3.008'}
{'loss': '0.01758', 'grad_norm': '0.007456', 'learning_rate': '3.12e-06', 'epoch': '3.383'}
{'loss': '0.06561', 'grad_norm': '0.02036', 'learning_rate': '1.241e-06', 'epoch': '3.759'}
{'train_runtime': '139.1', 'train_samples_per_second': '61.03', 'train_steps_per_second': '3.825', 'trai

In [ ]:
!pip install -q transformers

import torch
import random
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from google.colab import files
print("Please upload darkguard_split_v1.tsv:")
uploaded = files.upload()
split_df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")

trainval_df = split_df[split_df["split"] == "trainval"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

# --- Round 1: all 62 now fold into training (their OOD evaluation job is done) ---
round1_all = [
    "Price: $24.99", "Price: ₹3,499", "$49.00", "₹899", "Rs. 1,250.00",
    "Total: $128.45", "Subtotal: $56.20", "$12.99 per item", "MRP: ₹2,999", "Regular price $75.00",
    "Quantity: 1", "Quantity:", "Qty: 2", "Item quantity", "Select quantity",
    "Update quantity", "1 item in cart", "3 items in your bag",
    "Friday, 25 September", "Order placed on March 3, 2024", "Delivery by June 12",
    "Estimated arrival: 2-3 business days", "Return window: 30 days", "Available from January 15", "Ships on Monday",
    "Delivering to Dindigul 624005", "Ship to: New York, NY 10001", "123 Main Street, Apt 4B",
    "Delivery address", "Change delivery address", "Shipping to United States", "Estimated delivery to your area",
    "Order #48213", "Tracking number: 1Z999AA10123456784", "Invoice #2024-0091",
    "Account number ending in 4417", "Reference ID: 88213",
    "30% off", "50% discount", "Save 20%", "10% cashback on this order", "Tax: 8.5%", "Interest rate: 4.9% APR",
    "4.5 out of 5 stars", "Rated 4.2 based on 310 reviews", "3 reviews", "128 customers rated this 5 stars",
    "Size: 8 UK", "Weight: 2.5 kg", "Dimensions: 12 x 8 x 4 inches", "Available in sizes 6, 8, 10, 12",
    "500ml bottle", "Pack of 3",
    "Call us at 1-800-555-0199", "Customer service: (555) 123-4567",
    "Page 2 of 14", "Showing 1-24 of 350 results", "12 items per page",
    "Card ending in 4417", "Expires 08/27", "3 installments of $16.33", "Pay in 4 interest-free payments",
]

# --- Round 2: 21 go to training, 6 held out fresh for OOD testing ---
round2_train = [
    "Showing 25-48 of 512 results", "1-12 of 64 results", "Page 1 of 25", "View all 214 results", "Next page",
    "Ordered on April 18, 2023", "Your order was placed on November 9", "Delivered on September 14",
    "Return requested on July 1", "Estimated delivery: October 5-7",
    "2 items in your cart", "5 items in your bag", "0 items in cart", "Cart (3)",
    "Only 30% off",              # literal backend-failing phrase - direct fix
    "Only $5 off", "Only ₹200 off", "Only 2% cashback",
    "Sold by Amazon",            # literal backend-failing phrase - direct fix
    "Sold by ThirdPartySeller Inc.", "Fulfilled by Amazon",
]
round2_ood_holdout = [
    "Displaying 1-20 of 87 items", "Purchased on 12 May 2022", "You have 4 items saved",
    "Wishlist (7)", "Only 15% off select items", "Sold and shipped by Walmart",
]

aug_train_texts = round1_all + round2_train
aug_train_df = pd.DataFrame({
    "page_id": range(90000, 90000 + len(aug_train_texts)),
    "text": aug_train_texts,
    "label": 0,
    "Pattern Category": "Not Dark Pattern",
})

combined_train_df = pd.concat([
    trainval_df[["page_id", "text", "label", "Pattern Category"]],
    aug_train_df
], ignore_index=True)
print(f"Combined training pool: {len(combined_train_df)} rows "
      f"({len(trainval_df)} original + {len(round1_all)} round1 + {len(round2_train)} round2)")
print(f"Fresh OOD holdout (round 2, never trained on): {len(round2_ood_holdout)}")

MAX_LENGTH = 64
tok = AutoTokenizer.from_pretrained("roberta-large")

def encode(texts):
    return tok(list(texts), padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_encodings = encode(combined_train_df["text"])
test_encodings = encode(test_df["text"])
ood_encodings = encode(round2_ood_holdout)

backend_texts = [
    "Hurry! Only 1 left in stock!", "Only 4 left in stock.", "Only 2 items remaining!",
    "Selling fast! Only 3 left.", "23 people are viewing this right now.", "Offer ends in 10 minutes!",
    "LAST CHANCE - BUY NOW!", "Pillowcases & Shams", "Write a review", "International Shipping Policy",
    "Product Feature", "Sold by Amazon", "Quantity:", "Delivery date", "Friday, 25 September",
    "Delivering to Dindigul 624005", "3,499.00", "30% off", "50% discount", "Sale price ₹3,499",
    "Only 30% off", "Limited time 30% off", "30% off - offer ends tonight",
]
backend_labels = [1,1,1,1,1,1,1, 0,0,0,0,0,0,0,0,0,0,0,0,0,0, 1,1]
backend_encodings = encode(backend_texts)

class DarkPatternDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = DarkPatternDataset(train_encodings, list(combined_train_df["label"]))
test_dataset = DarkPatternDataset(test_encodings, list(test_df["label"]))
ood_dataset = DarkPatternDataset(ood_encodings, [0]*len(round2_ood_holdout))
backend_dataset = DarkPatternDataset(backend_encodings, backend_labels)

model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

training_args = TrainingArguments(
    output_dir="./roberta_large_augmented_r2",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,
    fp16=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, compute_metrics=compute_metrics)
trainer.train()

print("\n=== 1. ORIGINAL 284-row held-out test (regression check) ===")
print(trainer.evaluate(test_dataset))

print("\n=== 2. Round 2 FRESH held-out OOD examples (6, never trained on) ===")
ood_preds = np.argmax(trainer.predict(ood_dataset).predictions, axis=1)
for text, pred in zip(round2_ood_holdout, ood_preds):
    print(f"{'✓' if pred==0 else '✗'} predicted={pred} | {text}")
print(f"Fresh OOD accuracy: {(ood_preds==0).mean():.2%}")

print("\n=== 3. Backend team's 23 real examples ===")
backend_preds = np.argmax(trainer.predict(backend_dataset).predictions, axis=1)
correct = sum(p==e for p,e in zip(backend_preds, backend_labels))
for text, pred, expected in zip(backend_texts, backend_preds, backend_labels):
    print(f"{'✓' if pred==expected else '✗'} expected={expected} predicted={pred} | {text}")
print(f"Backend accuracy: {correct}/{len(backend_texts)} = {correct/len(backend_texts):.2%}")
print("(Round 1 result: 21/23 = 91.30% | Original pre-fix: 15/23 = 65.22%)")

Please upload darkguard_split_v1.tsv:


Saving darkguard_split_v1.tsv to darkguard_split_v1.tsv
Combined training pool: 2155 rows (2072 original + 62 round1 + 21 round2)
Fresh OOD holdout (round 2, never trained on): 6


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.9462', 'grad_norm': '15.08', 'learning_rate': '1.819e-05', 'epoch': '0.3704'}
{'loss': '0.4883', 'grad_norm': '87.94', 'learning_rate': '1.633e-05', 'epoch': '0.7407'}
{'loss': '0.4072', 'grad_norm': '24.54', 'learning_rate': '1.448e-05', 'epoch': '1.111'}
{'loss': '0.2379', 'grad_norm': '12.76', 'learning_rate': '1.263e-05', 'epoch': '1.481'}
{'loss': '0.4592', 'grad_norm': '0.4213', 'learning_rate': '1.078e-05', 'epoch': '1.852'}
{'loss': '0.1035', 'grad_norm': '11.39', 'learning_rate': '8.926e-06', 'epoch': '2.222'}
{'loss': '0.1452', 'grad_norm': '0.2114', 'learning_rate': '7.074e-06', 'epoch': '2.593'}
{'loss': '0.2571', 'grad_norm': '0.02985', 'learning_rate': '5.222e-06', 'epoch': '2.963'}
{'loss': '0.123', 'grad_norm': '0.02564', 'learning_rate': '3.37e-06', 'epoch': '3.333'}
{'loss': '0.03163', 'grad_norm': '0.05303', 'learning_rate': '1.519e-06', 'epoch': '3.704'}
{'train_runtime': '144.4', 'train_samples_per_second': '59.69', 'train_steps_per_second': '3.739', 't

In [ ]:
!pip install -q transformers

import torch
import pandas as pd
from transformers import AutoTokenizer, RobertaForSequenceClassification, TrainingArguments, Trainer

from google.colab import files
print("Please upload darkguard_split_v1.tsv:")
uploaded = files.upload()
split_df = pd.read_csv("darkguard_split_v1.tsv", sep="\t")
all_original = split_df[["page_id", "text", "label", "Pattern Category"]].copy()  # trainval + test, everything

all_augmentation = [
    # Round 1 (62)
    "Price: $24.99", "Price: ₹3,499", "$49.00", "₹899", "Rs. 1,250.00",
    "Total: $128.45", "Subtotal: $56.20", "$12.99 per item", "MRP: ₹2,999", "Regular price $75.00",
    "Quantity: 1", "Quantity:", "Qty: 2", "Item quantity", "Select quantity",
    "Update quantity", "1 item in cart", "3 items in your bag",
    "Friday, 25 September", "Order placed on March 3, 2024", "Delivery by June 12",
    "Estimated arrival: 2-3 business days", "Return window: 30 days", "Available from January 15", "Ships on Monday",
    "Delivering to Dindigul 624005", "Ship to: New York, NY 10001", "123 Main Street, Apt 4B",
    "Delivery address", "Change delivery address", "Shipping to United States", "Estimated delivery to your area",
    "Order #48213", "Tracking number: 1Z999AA10123456784", "Invoice #2024-0091",
    "Account number ending in 4417", "Reference ID: 88213",
    "30% off", "50% discount", "Save 20%", "10% cashback on this order", "Tax: 8.5%", "Interest rate: 4.9% APR",
    "4.5 out of 5 stars", "Rated 4.2 based on 310 reviews", "3 reviews", "128 customers rated this 5 stars",
    "Size: 8 UK", "Weight: 2.5 kg", "Dimensions: 12 x 8 x 4 inches", "Available in sizes 6, 8, 10, 12",
    "500ml bottle", "Pack of 3", "Call us at 1-800-555-0199", "Customer service: (555) 123-4567",
    "Page 2 of 14", "Showing 1-24 of 350 results", "12 items per page",
    "Card ending in 4417", "Expires 08/27", "3 installments of $16.33", "Pay in 4 interest-free payments",
    # Round 2, all 27 including the former OOD holdout
    "Showing 25-48 of 512 results", "Displaying 1-20 of 87 items", "1-12 of 64 results", "Page 1 of 25",
    "View all 214 results", "Next page",
    "Ordered on April 18, 2023", "Purchased on 12 May 2022", "Your order was placed on November 9",
    "Delivered on September 14", "Return requested on July 1", "Estimated delivery: October 5-7",
    "2 items in your cart", "5 items in your bag", "You have 4 items saved", "0 items in cart",
    "Cart (3)", "Wishlist (7)",
    "Only 30% off", "Only 15% off select items", "Only $5 off", "Only ₹200 off", "Only 2% cashback",
    "Sold by Amazon", "Sold by ThirdPartySeller Inc.", "Sold and shipped by Walmart", "Fulfilled by Amazon",
]

aug_df = pd.DataFrame({
    "page_id": range(90000, 90000 + len(all_augmentation)),
    "text": all_augmentation,
    "label": 0,
    "Pattern Category": "Not Dark Pattern",
})

full_final_df = pd.concat([all_original, aug_df], ignore_index=True)
print(f"FINAL training set: {len(full_final_df)} rows "
      f"({len(all_original)} original [train+val+test] + {len(all_augmentation)} augmentation)")

MAX_LENGTH = 64
tok = AutoTokenizer.from_pretrained("roberta-large")
encodings = tok(list(full_final_df["text"]), padding="max_length", truncation=True, max_length=MAX_LENGTH)

class DarkPatternDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

full_dataset = DarkPatternDataset(encodings, list(full_final_df["label"]))

final_model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

training_args = TrainingArguments(
    output_dir="./roberta_large_deployment_v2",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    seed=42,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
    disable_tqdm=True,
    fp16=True,
)

trainer = Trainer(model=final_model, args=training_args, train_dataset=full_dataset)
trainer.train()

final_model.save_pretrained("./darkguard_deployment_model_v2")
tok.save_pretrained("./darkguard_deployment_model_v2")

import shutil
shutil.make_archive("darkguard_roberta_large_final_v2", "zip", "./darkguard_deployment_model_v2")
print("Saved: darkguard_roberta_large_final_v2.zip")

files.download("darkguard_roberta_large_final_v2.zip")

Please upload darkguard_split_v1.tsv:


Saving darkguard_split_v1.tsv to darkguard_split_v1 (1).tsv
FINAL training set: 2445 rows (2356 original [train+val+test] + 89 augmentation)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.9093', 'grad_norm': '93.55', 'learning_rate': '1.84e-05', 'epoch': '0.3268'}
{'loss': '0.5538', 'grad_norm': '131', 'learning_rate': '1.676e-05', 'epoch': '0.6536'}
{'loss': '0.3476', 'grad_norm': '8.717', 'learning_rate': '1.513e-05', 'epoch': '0.9804'}
{'loss': '0.3303', 'grad_norm': '4.863', 'learning_rate': '1.35e-05', 'epoch': '1.307'}
{'loss': '0.2732', 'grad_norm': '0.07271', 'learning_rate': '1.186e-05', 'epoch': '1.634'}
{'loss': '0.1961', 'grad_norm': '0.1219', 'learning_rate': '1.023e-05', 'epoch': '1.961'}
{'loss': '0.1116', 'grad_norm': '0.0178', 'learning_rate': '8.595e-06', 'epoch': '2.288'}
{'loss': '0.116', 'grad_norm': '0.1357', 'learning_rate': '6.961e-06', 'epoch': '2.614'}
{'loss': '0.1244', 'grad_norm': '0.04935', 'learning_rate': '5.327e-06', 'epoch': '2.941'}
{'loss': '0.1372', 'grad_norm': '0.05007', 'learning_rate': '3.693e-06', 'epoch': '3.268'}
{'loss': '0.06627', 'grad_norm': '0.01653', 'learning_rate': '2.059e-06', 'epoch': '3.595'}
{'loss': '0

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: darkguard_roberta_large_final_v2.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>